In [43]:
pip install sentence-transformers qdrant-client

In [1]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

from qdrant_client import QdrantClient

from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct
)

import warnings
warnings.filterwarnings("ignore")

In [4]:
df = pd.read_csv('C:/Users/Juilee/Desktop/Big data and Bussiness Intelligence Capstone Project/data/merged_df.csv')
print("Dataset Loaded Successfully") 
print("\nDataset Shape:\n") 
print(df.shape) 
print("\nFirst 5 Rows:\n") 
display(df.head()) 
print("\nColumns:\n") 
print(df.columns.tolist())

Dataset Loaded Successfully

Dataset Shape:

(180519, 23)

First 5 Rows:



,order_id,customer_id,product_name,category,department,Product Price,product_image,Order Region,Market,Order Status,...,sales,quantity,profit,Latitude,Longitude,delay,view_count,unique_users,peak_month,peak_hour
0,77202,20755,smart watch,sporting goods,Fitness,327.75,http://images.acmesports.sports/Smart+watch,Southeast Asia,Pacific Asia,COMPLETE,...,327.75,1,91.250000,18.251453,-66.037056,-1.0,0.0,0.0,NaN,NaN
1,75939,19492,smart watch,sporting goods,Fitness,327.75,http://images.acmesports.sports/Smart+watch,South Asia,Pacific Asia,PENDING,...,327.75,1,-249.089996,18.279451,-66.037064,1.0,0.0,0.0,NaN,NaN
2,75938,19491,smart watch,sporting goods,Fitness,327.75,http://images.acmesports.sports/Smart+watch,South Asia,Pacific Asia,CLOSED,...,327.75,1,-247.779999,37.292233,-121.881279,0.0,0.0,0.0,NaN,NaN
3,75937,19490,smart watch,sporting goods,Fitness,327.75,http://images.acmesports.sports/Smart+watch,Oceania,Pacific Asia,COMPLETE,...,327.75,1,22.860001,34.125946,-118.291016,-1.0,0.0,0.0,NaN,NaN
4,75936,19489,smart watch,sporting goods,Fitness,327.75,http://images.acmesports.sports/Smart+watch,Oceania,Pacific Asia,PENDING_PAYMENT,...,327.75,1,134.210007,18.253769,-66.037048,-2.0,0.0,0.0,NaN,NaN



Columns:

['order_id', 'customer_id', 'product_name', 'category', 'department', 'Product Price', 'product_image', 'Order Region', 'Market', 'Order Status', 'Shipping Mode', 'actual_days', 'scheduled_days', 'sales', 'quantity', 'profit', 'Latitude', 'Longitude', 'delay', 'view_count', 'unique_users', 'peak_month', 'peak_hour']


In [5]:
df["text_data"] = ( df["product_name"].fillna("").astype(str) + " " + df["category"].fillna("").astype(str) + " " + df["department"].fillna("").astype(str) )

In [7]:
df = df[[ "order_id", "text_data" ]] 
df = df[ df["text_data"].str.strip() != "" ]

In [8]:
df = df.reset_index(drop=True) 
print("Text Data Created Successfully") 
print("\nDataset Shape:\n") 
print(df.shape) 
display(df.head())

Text Data Created Successfully

Dataset Shape:

(180519, 2)


,order_id,text_data
0,77202,smart watch sporting goods Fitness
1,75939,smart watch sporting goods Fitness
2,75938,smart watch sporting goods Fitness
3,75937,smart watch sporting goods Fitness
4,75936,smart watch sporting goods Fitness


In [9]:
model = SentenceTransformer( "all-MiniLM-L6-v2" ) 
print("SentenceTransformer Model Loaded Successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

SentenceTransformer Model Loaded Successfully


In [10]:
texts = df["text_data"].tolist() 
embeddings = model.encode( texts, show_progress_bar=True ) 
print("Embeddings Generated Successfully") 
print("\nEmbedding Shape:\n") 
print(embeddings.shape)

Batches:   0%|          | 0/5642 [00:00<?, ?it/s]

Embeddings Generated Successfully

Embedding Shape:

(180519, 384)


In [11]:
client = QdrantClient(":memory:")

print("Connected to Local Qdrant Successfully")

Connected to Local Qdrant Successfully


In [12]:
COLLECTION_NAME = "capstone_embeddings"


if client.collection_exists(COLLECTION_NAME):

    client.delete_collection(COLLECTION_NAME)

    print("Old Collection Deleted")


client.create_collection(

    collection_name=COLLECTION_NAME,

    vectors_config=VectorParams(

        size=384,

        distance=Distance.COSINE
    )
)

print("Qdrant Collection Created Successfully")

Qdrant Collection Created Successfully


In [14]:
points = []

for idx, row in df.iterrows():

    point = PointStruct(

        id=int(idx),

        vector=embeddings[idx].tolist(),

        payload={

            "order_id": str(row["order_id"]),

            "raw_text": row["text_data"]
        }
    )

    points.append(point)

print("Points Created Successfully")

print("\nTotal Points:\n")

print(len(points))

Points Created Successfully

Total Points:

180519


In [15]:

BATCH_SIZE = 1000

for i in range(0, len(points), BATCH_SIZE):

    batch = points[i:i+BATCH_SIZE]

    client.upsert(

        collection_name=COLLECTION_NAME,

        points=batch
    )

print("Embeddings Uploaded Successfully")

Embeddings Uploaded Successfully


In [16]:
query_text = "running shoes for fitness training"

print("\nQuery Text:\n")

print(query_text)


Query Text:

running shoes for fitness training


In [17]:
query_vector = model.encode(
    query_text
).tolist()

print("Query Embedding Generated Successfully")

Query Embedding Generated Successfully


In [18]:
results = client.query_points(

    collection_name=COLLECTION_NAME,

    query=query_vector,

    limit=5,

    with_payload=True
)

print("Similarity Search Completed Successfully")

Similarity Search Completed Successfully


In [20]:
print("\nTOP 5 SIMILAR RESULTS\n")

for idx, hit in enumerate(results.points, start=1):

    print("=" * 70)

    print(f"RESULT {idx}")

    print(f"\nSimilarity Score: {round(hit.score, 4)}")

    print(f"\nNode ID: {hit.payload['order_id']}")

    print("\nRetrieved Text:")

    print(hit.payload["raw_text"])

    print("=" * 70)


TOP 5 SIMILAR RESULTS

RESULT 1

Similarity Score: 0.6172

Node ID: 12827

Retrieved Text:
nike men's free 5.0+ running shoe cardio equipment Footwear
RESULT 2

Similarity Score: 0.6172

Node ID: 57369

Retrieved Text:
nike men's free 5.0+ running shoe cardio equipment Footwear
RESULT 3

Similarity Score: 0.6172

Node ID: 2203

Retrieved Text:
nike men's free 5.0+ running shoe cardio equipment Footwear
RESULT 4

Similarity Score: 0.6172

Node ID: 49572

Retrieved Text:
nike men's free 5.0+ running shoe cardio equipment Footwear
RESULT 5

Similarity Score: 0.6172

Node ID: 41181

Retrieved Text:
nike men's free 5.0+ running shoe cardio equipment Footwear


In [22]:
embedding_df = pd.DataFrame(embeddings)


embedding_df["order_id"] = df["order_id"]

embedding_df.to_parquet(
    "C:/Users/Juilee/Desktop/Big data and Bussiness Intelligence Capstone Project/data/product_embeddings.parquet",
    index=False
)

print("Embeddings Saved Successfully")

Embeddings Saved Successfully


In [23]:
df.to_parquet(
    "C:/Users/Juilee/Desktop/Big data and Bussiness Intelligence Capstone Project/data/embedding_dataset.parquet",
    index=False
)

print("Embedding Dataset Saved Successfully")

Embedding Dataset Saved Successfully


In [32]:

print("\nOPERATIONS COMPLETED:")

print("- Loaded Dataset")
print("- Created Text Field")
print("- Generated Sentence Embeddings")
print("- Connected to Qdrant")
print("- Created Collection")
print("- Uploaded Embeddings")
print("- Performed Similarity Search")

print("\nMODEL USED:")

print("- all-MiniLM-L6-v2")

print("\nVECTOR DATABASE:")

print("- Qdrant")
print("- COSINE Similarity")
print("- 384-dimensional vectors")

print("\nRAG STATUS:")

print("- Retrieval COMPLETE")
print("- Augmentation READY")
print("- Generation conceptual only")


OPERATIONS COMPLETED:
- Loaded Dataset
- Created Text Field
- Generated Sentence Embeddings
- Connected to Qdrant
- Created Collection
- Uploaded Embeddings
- Performed Similarity Search

MODEL USED:
- all-MiniLM-L6-v2

VECTOR DATABASE:
- Qdrant
- COSINE Similarity
- 384-dimensional vectors

RAG STATUS:
- Retrieval COMPLETE
- Augmentation READY
- Generation conceptual only


In [28]:
df.columns

Index(['order_id', 'text_data'], dtype='object')

In [31]:
embeddings_df = pd.DataFrame({

    "order_id": df["order_id"],

    "raw_text": df["text_data"],

    "embedding": embeddings.tolist()
})

embeddings_df.to_parquet(
    "C:/Users/Juilee/Desktop/Big data and Bussiness Intelligence Capstone Project/data/product_embeddings.parquet",
    index=False
)

print("Embeddings parquet saved successfully.")

Embeddings parquet saved successfully.
